# Matrix Factorisation on Movie Lens 1M dataset
Dataset from: [Movie Lens 1M Dataset](https://grouplens.org/datasets/movielens/1m/)

In [1]:
# Standard
import pandas as pd
import re

# Third-party
import numpy as np
import plotly.express as px
from sklearn.metrics import explained_variance_score, mean_squared_error, mean_absolute_error, r2_score

# Local
from Dataset.ml1m_data_loader import load_and_merge_data, ml_test_train_split
from models.matrix_factorisation.NMF_matrix_factorisation import NMFMatrixFactorisation, create_pivot_table

### Load dataset

In [2]:
df = load_and_merge_data()
df_train, df_test = ml_test_train_split(df, test_proportion=0.2)

In [3]:
df_train.head()

,user_id,movie_id,rating,timestamp,title,genres
0,392,587,1,2000-12-08 19:43:44,Ghost (1990),Comedy|Romance|Thriller
1,1449,2053,1,2000-11-28 18:49:33,"Honey, I Blew Up the Kid (1992)",Children's|Comedy|Sci-Fi
2,2856,1290,3,2000-10-25 20:22:00,Some Kind of Wonderful (1987),Drama|Romance
3,808,2279,3,2000-11-28 06:39:46,Urban Legend (1998),Horror|Thriller
4,4480,2138,4,2000-07-31 06:32:45,Watership Down (1978),Animation|Children's|Drama|Fantasy


### Matrix Factorisation
Matrix factorisation algorithm applied to Top-N and Similarity (by movie) slates.

i.e. answers the questions: "what are the top N movies for a specific user" and "because someone watched a movie, they should watch"


Using:
- [SKLearn NMF (Non-Negative Matrix Factorisation)](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.NMF.html)]
- Similarity code from [here](https://github.com/dinesh-git17/movie_recommendation/tree/main)
- Top-N code from [here](https://medium.com/@quindaly/step-by-step-nmf-example-in-python-9974e38dc9f9)

In [4]:
NMF_model = NMFMatrixFactorisation(df_train, n_components=50)

### Evaluation metrics
Offline metrics - adapted from [here](https://github.com/aryan-jadon/Evaluation-Metrics-for-Recommendation-Systems/blob/main/recommenders/evaluation/python_evaluation.py)

Table from [here](https://github.com/recommenders-team/recommenders/blob/main/examples/03_evaluate/evaluation.ipynb)
|Metric|Range|Selection criteria|Limitation|Reference|
|------|-------------------------------|---------|----------|---------|
|RMSE|$> 0$|The smaller the better.|May be biased, and less explainable than MAE|[link](https://en.wikipedia.org/wiki/Root-mean-square_deviation)|
|MAE|$\geq 0$|The smaller the better.|Dependent on variable scale.|[link](https://en.wikipedia.org/wiki/Mean_absolute_error)|
|R2|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Coefficient_of_determination)|
|Explained variance|$\leq 1$|The closer to $1$ the better.|Depend on variable distributions.|[link](https://en.wikipedia.org/wiki/Explained_variation)|

In [5]:
def evaluation_metrics(y_true, y_pred):
    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R Squared": r2_score(y_true, y_pred),
        "Explained variance": explained_variance_score(y_true, y_pred)
    }

In [6]:
print("Metrics on training data:")
[print(f"{k}: {v:.2f}") for k, v in evaluation_metrics(NMF_model.pivot, NMF_model.V).items()]

Metrics on training data:
RMSE: 0.75
MAE: 0.32
R Squared: 0.24
Explained variance: 0.24


[None, None, None, None]

In [7]:
print("Metrics on test data:")
# Make test pivot table
test_pivot = create_pivot_table(df_test)
# Extract matching sections of predicted pivot table

# Compare

Metrics on test data:


In [ ]:
# OR
# - get predicted ratings returned with recs
# - get predicted ratings for movie somehow? without recommendation list?



### Apply model

In [5]:
recs = NMF_model.movie_similarity("101 Dalmatians (1961)")
recs

Bambi (1942)                  0.996216
Aristocats, The (1970)        0.996108
Dumbo (1941)                  0.990570
Pinocchio (1940)              0.990138
Song of the South (1946)      0.988481
Robin Hood (1973)             0.987298
Alice in Wonderland (1951)    0.986991
Rescuers, The (1977)          0.986478
Cinderella (1950)             0.986174
Jungle Book, The (1967)       0.986111
Name: 101 Dalmatians (1961), dtype: float64

In [6]:
recs = NMF_model.movie_similarity("101 Dalmatians (1996)")
recs

Matilda (1996)                                   0.954150
Paulie (1998)                                    0.934353
George of the Jungle (1997)                      0.931422
Homeward Bound: The Incredible Journey (1993)    0.898202
Casper (1995)                                    0.890479
Mouse Hunt (1997)                                0.883225
Parent Trap, The (1998)                          0.867603
Fly Away Home (1996)                             0.863836
Angels in the Outfield (1994)                    0.851627
American Tail: Fievel Goes West, An (1991)       0.831820
Name: 101 Dalmatians (1996), dtype: float64

In [7]:
recs = NMF_model.movie_similarity("10 Things I Hate About You (1999)")
recs

She's All That (1999)                0.952949
Never Been Kissed (1999)             0.946906
Story of Us, The (1999)              0.922726
Mickey Blue Eyes (1999)              0.916923
Bachelor, The (1999)                 0.909128
Random Hearts (1999)                 0.908257
Love Letter, The (1999)              0.895853
Deep End of the Ocean, The (1999)    0.893656
Runaway Bride (1999)                 0.865351
For Love of the Game (1999)          0.859388
Name: 10 Things I Hate About You (1999), dtype: float64

In [8]:
recs = NMF_model.movie_similarity("Young Guns (1988)")
recs

Quick and the Dead, The (1995)    0.914251
Young Guns II (1990)              0.853148
Last Man Standing (1996)          0.847203
Maverick (1994)                   0.775005
Wyatt Earp (1994)                 0.760288
Pale Rider (1985)                 0.758648
Outlaw Josey Wales, The (1976)    0.758504
Hang 'em High (1967)              0.753177
High Plains Drifter (1972)        0.751716
For a Few Dollars More (1965)     0.751429
Name: Young Guns (1988), dtype: float64

Thoughts:
* Not recommending sequels
* Not using a test-train split -> this algorithm won't work if the requested movie doesn't exist in the pivot table
* Therefore, can't handle new movies or users

## Top-N movies for user

In [ ]:
user_id  = 44
NMF_model.understand_user_profile(user_id)
rec = NMF_model.user_top_N(user_id)
print(f"Recommendations for user 44:")
display(NMF_model.get_recommend_dataframe(rec))

## User profile evaluation

In [8]:
user_ids = [6013, 2195, 1198, 3662, 4713]

In [9]:
for user_id in user_ids:
    NMF_model.understand_user_profile(user_id, rating_dist=False, wc=False)
    rec = NMF_model.user_top_N(user_id)
    print(f"Recommendations for user {user_id}")
    display(NMF_model.get_recommend_dataframe(rec))

Recommendations for user 6013


,title,genres
0,Casablanca (1942),"[Drama, Romance, War]"
1,Chicken Run (2000),"[Animation, Children's, Comedy]"
2,Saving Private Ryan (1998),"[Action, Drama, War]"
3,"Grand Day Out, A (1992)","[Animation, Comedy]"
4,Toy Story 2 (1999),"[Animation, Children's, Comedy]"
5,Fantasia (1940),"[Animation, Children's, Musical]"
6,"Bug's Life, A (1998)","[Animation, Children's, Comedy]"
7,Aladdin (1992),"[Animation, Children's, Comedy, Musical]"
8,Star Wars: Episode V - The Empire Strikes Back...,"[Action, Adventure, Drama, Sci-Fi, War]"
9,Gone with the Wind (1939),"[Drama, Romance, War]"


Recommendations for user 2195


,title,genres
0,Star Wars: Episode I - The Phantom Menace (1999),"[Action, Adventure, Fantasy, Sci-Fi]"
1,Star Trek IV: The Voyage Home (1986),"[Action, Adventure, Sci-Fi]"
2,Indiana Jones and the Temple of Doom (1984),"[Action, Adventure]"
3,E.T. the Extra-Terrestrial (1982),"[Children's, Drama, Fantasy, Sci-Fi]"
4,Starship Troopers (1997),"[Action, Adventure, Sci-Fi, War]"
5,Twelve Monkeys (1995),"[Drama, Sci-Fi]"
6,"X-Files: Fight the Future, The (1998)","[Mystery, Sci-Fi, Thriller]"
7,Sneakers (1992),"[Crime, Drama, Sci-Fi]"
8,Romancing the Stone (1984),"[Action, Adventure, Comedy, Romance]"
9,Total Recall (1990),"[Action, Adventure, Sci-Fi, Thriller]"


Recommendations for user 1198


,title,genres
0,X-Men (2000),"[Action, Sci-Fi]"
1,Star Wars: Episode V - The Empire Strikes Back...,"[Action, Adventure, Drama, Sci-Fi, War]"
2,"Matrix, The (1999)","[Action, Sci-Fi, Thriller]"
3,Jurassic Park (1993),"[Action, Adventure, Sci-Fi]"
4,Frequency (2000),"[Drama, Thriller]"
5,Indiana Jones and the Last Crusade (1989),"[Action, Adventure]"
6,"Fugitive, The (1993)","[Action, Thriller]"
7,Gone in 60 Seconds (2000),"[Action, Crime]"
8,Erin Brockovich (2000),[Drama]
9,Fight Club (1999),[Drama]


Recommendations for user 3662


,title,genres
0,"Thing, The (1982)","[Action, Horror, Sci-Fi, Thriller]"
1,"Fly, The (1986)","[Horror, Sci-Fi]"
2,Escape from New York (1981),"[Action, Adventure, Sci-Fi, Thriller]"
3,"Shining, The (1980)",[Horror]
4,Westworld (1973),"[Action, Sci-Fi, Thriller, Western]"
5,Predator (1987),"[Action, Sci-Fi, Thriller]"
6,Mad Max 2 (a.k.a. The Road Warrior) (1981),"[Action, Sci-Fi]"
7,Soylent Green (1973),"[Sci-Fi, Thriller]"
8,Logan's Run (1976),"[Action, Adventure, Sci-Fi]"
9,"Exorcist, The (1973)",[Horror]


Recommendations for user 4713


,title,genres
0,Schindler's List (1993),"[Drama, War]"
1,"Godfather, The (1972)","[Action, Crime, Drama]"
2,X-Men (2000),"[Action, Sci-Fi]"
3,"Perfect Storm, The (2000)","[Action, Adventure, Thriller]"
4,U-571 (2000),"[Action, Thriller]"
5,Erin Brockovich (2000),[Drama]
6,Sleepless in Seattle (1993),"[Comedy, Romance]"
7,"Green Mile, The (1999)","[Drama, Thriller]"
8,Pretty Woman (1990),"[Comedy, Romance]"
9,Groundhog Day (1993),"[Comedy, Romance]"
